## Merged Features (Cross-Table)

Now combining cleaned and feature-enriched tables to create master datasets and derive features that require information from multiple sources.

In [1]:
import pandas as pd

In [2]:
orders = pd.read_csv("../data/feature_engineered/orders_fe.csv")
order_items = pd.read_csv("../data/feature_engineered/order_items_fe.csv")
customers = pd.read_csv("../data/feature_engineered/customers_fe.csv")
sellers = pd.read_csv("../data/feature_engineered/sellers_fe.csv")
products = pd.read_csv("../data/feature_engineered/products_fe.csv")
order_payments = pd.read_csv("../data/feature_engineered/order_payments_fe.csv")
order_reviews = pd.read_csv("../data/feature_engineered/order_reviews_fe.csv")
closed_deals = pd.read_csv("../data/feature_engineered/closed_deals_fe.csv")
marketing_leads = pd.read_csv("../data/feature_engineered/marketing_leads_fe.csv")

product_category_name_translations = pd.read_csv(
    "../data/processed/category_translation_clean.csv"
)
geo_locations = pd.read_csv("../data/processed/geolocation_clean.csv")

Before directly performing a merge, I verify the grains and keys of the tables I have, because this is fundamental to a successful merge. This ensures that the feature engineer tables I previously saved are also read correctly.

In [3]:
tables = {
    "orders": orders,
    "order_items": order_items,
    "customers": customers,
    "sellers": sellers,
    "products": products,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "closed_deals": closed_deals,
    "marketing_leads": marketing_leads,
    "geo_locations": geo_locations,
    "category_translation": product_category_name_translations
}

for name, df in tables.items():
    print(f"{name}: {df.shape}")

orders: (99441, 20)
order_items: (112650, 16)
customers: (99441, 7)
sellers: (3095, 6)
products: (32951, 12)
order_payments: (103886, 5)
order_reviews: (98673, 10)
closed_deals: (842, 18)
marketing_leads: (8000, 8)
geo_locations: (1000163, 5)
category_translation: (71, 2)


I also check if the keys are truly unique. Apart from that, I don't expect order_items, order_payments, and order_reviews to be unique because they are in different grains.

In [4]:
print("orders order_id unique:", orders["order_id"].is_unique)
print("customers customer_id unique:", customers["customer_id"].is_unique)
print("sellers seller_id unique:", sellers["seller_id"].is_unique)
print("products product_id unique:", products["product_id"].is_unique)
print("marketing_leads mql_id unique:", marketing_leads["mql_id"].is_unique)
print("closed_deals mql_id unique:", closed_deals["mql_id"].is_unique)

orders order_id unique: True
customers customer_id unique: True
sellers seller_id unique: True
products product_id unique: True
marketing_leads mql_id unique: True
closed_deals mql_id unique: True


In [5]:
orders_before = len(orders)
customers_before = len(customers)

print("Orders:", orders_before)
print("Customers:", customers_before)

Orders: 99441
Customers: 99441


In [6]:
print(
    "Orders with customer_id:",
    orders["customer_id"].notna().sum()
)

print(
    "Unique customer_ids in orders:",
    orders["customer_id"].nunique()
)

Orders with customer_id: 99441
Unique customer_ids in orders: 99441


In [7]:
df_master = orders.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

In [8]:
print("Before:", len(orders))
print("After:", len(df_master))

Before: 99441
After: 99441


After the orders and customers tables were merged, the number of columns in the df_master table increased, as expected.

In [9]:
print(df_master.shape)

(99441, 26)


Now, I will also bind the necessary properties related to the "order_items" table to this df_master variable. If I used a different variable name for each merge operation, there would be many tables and it would look very messy. So I continue via "df_master".
First, I examine the order_items table.

In [10]:
order_items.columns

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value', 'item_total_cost',
       'freight_ratio', 'items_in_order', 'order_total_price',
       'order_total_freight', 'order_total_cost', 'sellers_in_order',
       'is_free_shipping', 'average_item_price'],
      dtype='str')

But in this case, directly merging the `order_items` table with `df_master` is an incorrect approach. If an order has 3 items, the row in `df_master` will be repeated 3 times. This results in order-level information, such as customer information and order details, being repeated row by row. This corrupts the `grain` of `df_master` and causes the "1 row = 1 order" property to be lost.

In [11]:
order_items.shape

(112650, 16)

Therefore, instead of directly linking the `order_items` table, I'm creating an `order_item_summary` variable. I'm also creating some of the features already present in the `order_items` table here. This is because I want to convert the `item-grain` data into an `order-grain` summary table. This also reduces the number of rows.

In [12]:
order_item_summary = (
    order_items
    .groupby("order_id")
    .agg(
        items_in_order=("order_item_id", "count"),
        order_total_price=("price", "sum"),
        order_total_freight=("freight_value", "sum"),
        order_total_cost=("item_total_cost", "sum"),
        sellers_in_order=("seller_id", "nunique"),
        is_free_shipping=("freight_value", lambda x: (x == 0).all()),
        average_item_price=("price", "mean")
    )
    .reset_index()
)

In [13]:
print(order_item_summary.shape)
print(order_item_summary["order_id"].nunique())

(98666, 8)
98666


After the merge operation, the properties from the order_item_summary table are also added to df_master, and the number of columns naturally increases.

In [14]:
df_master = df_master.merge(
    order_item_summary,
    on="order_id",
    how="left",
    validate="one_to_one"
)

print(df_master.shape)
print(df_master["order_id"].nunique())


(99441, 33)
99441


After merging the `order_item_summary` with `df_master`, I checked the newly added order-level features for missing values.

Since `df_master` was merged using a **left join**, these missing values indicate that 775 orders in `df_master` do not have a matching record in `order_item_summary`.

This is important to investigate later, but the missing values should not be filled with arbitrary values at this stage. First, the remaining order-level tables will be integrated and the reasons for these missing records will be evaluated.

In [15]:
df_master[
    [
        "items_in_order",
        "order_total_price",
        "order_total_freight",
        "order_total_cost",
        "sellers_in_order",
        "is_free_shipping",
        "average_item_price"
    ]
].isna().sum()

items_in_order         775
order_total_price      775
order_total_freight    775
order_total_cost       775
sellers_in_order       775
is_free_shipping       775
average_item_price     775
dtype: int64

Similarly, I'm examining the order_payments table. Looking at the row counts and the number of unique order_id entries, I see that there can be multiple payment records for a single order. In other words, the grain of this table is "1 row = 1 payment record". Again, I need to make this table compliant with df_master.

In [16]:
print(order_payments.shape)
print(order_payments["order_id"].nunique())

(103886, 5)
99440


In [17]:
payment_summary = (
    order_payments
    .groupby("order_id")
    .agg(
        total_payment_value=("payment_value", "sum"),
        payment_count=("payment_sequential", "count")
    )
    .reset_index()
)

In [18]:
print(payment_summary.shape)
print(payment_summary["order_id"].nunique())

(99440, 3)
99440


In [20]:
df_master = df_master.merge(
    payment_summary,
    on="order_id",
    how="left",
    validate="one_to_one"
)

The `payment_summary` table has 3 columns, one of which is `order_id` and is shared with `df_master`. Therefore, the number of columns in the `df_master` table will be updated to 33 + 2.

In [21]:
print(df_master.shape)
print(df_master["order_id"].nunique())

(99441, 35)
99441


The `order_reviews` table has a grain of **1 row = 1 order**, as `order_id` is unique after the cleaning process. Therefore, unlike `order_items` or `order_payments`, no additional aggregation is required before merging it with `df_master`.
Since `df_master` also has a grain of **1 row = 1 order**, the two tables can be safely merged using a one-to-one relationship.

In [22]:
print(order_reviews.shape)
print(order_reviews["order_id"].nunique())

(98673, 10)
98673


In [25]:
df_master = df_master.merge(
    order_reviews,
    on="order_id",
    how="left",
    validate="one_to_one"
)

print(df_master.shape)
print(df_master["order_id"].nunique())

(99441, 44)
99441


After merging the review data, the review-related columns were checked for missing values.

There are 768 orders without a corresponding review record. These missing values are expected because not every order has a corresponding review.

The comment-related columns have substantially more missing values because a customer may have submitted a review without leaving a written comment. Therefore, missing comment text does not necessarily mean that the review itself is missing.


In [28]:
df_master[
    [
        "review_score",
        "review_comment_title",
        "review_comment_message",
        "review_creation_date",
        "review_answer_timestamp",
        "response_time_hours",
        "has_comment",
        "review_length"
    ]
].isna().sum()

review_score                 768
review_comment_title       87889
review_comment_message     58665
review_creation_date         768
review_answer_timestamp      768
response_time_hours          768
has_comment                  768
review_length                768
dtype: int64

The `closed_deals` table is related to `marketing_leads` through `mql_id`. All 842 `mql_id` values in `closed_deals` have a corresponding record in `marketing_leads`.

Therefore, each closed deal can be linked to exactly one marketing lead, while not every marketing lead has a closed deal.

In [34]:
print(marketing_leads["mql_id"].nunique())
print(closed_deals["mql_id"].nunique())

print(closed_deals["mql_id"].isin(marketing_leads["mql_id"]).sum())

8000
842
842


In [35]:
marketing_leads["mql_id"].duplicated().sum()

np.int64(0)

In [36]:
closed_deals["mql_id"].duplicated().sum()

np.int64(0)

Because the purpose of this table is to analyze the marketing and sales funnel, all marketing leads should be preserved, including those that did not result in a closed deal.

Now the grain of `funnel_master` is: 1 row = 1 marketing lead (`mql_id`)

In [37]:
funnel_master = marketing_leads.merge(
    closed_deals,
    on="mql_id",
    how="left",
    validate="one_to_one"
)

print(funnel_master.shape)
print(funnel_master["mql_id"].nunique())

(8000, 25)
8000


After integrating `marketing_leads` and `closed_deals`, the `funnel_master` contains one row per marketing lead.

The next step was to enrich the funnel data with seller-level information from the `sellers` table. Before performing the merge, the relationship between `closed_deals` and `sellers` was checked through `seller_id`.

- `sellers` contains 3,095 unique `seller_id` values.
- `sellers.seller_id` contains no duplicates.
- `closed_deals` contains 842 unique `seller_id` values.
- 380 of the 842 seller IDs in `closed_deals` have a corresponding record in `sellers`.
- The remaining 462 seller IDs do not appear in the `sellers` table.


In [39]:
print(sellers.shape)
print(sellers["seller_id"].nunique())
print(sellers["seller_id"].duplicated().sum())

(3095, 6)
3095
0


In [40]:
print(
    closed_deals["seller_id"].isin(sellers["seller_id"]).sum()
)

380


In [41]:
print(
    closed_deals.loc[
        ~closed_deals["seller_id"].isin(sellers["seller_id"]),
        "seller_id"
    ].nunique()
)

462



I also checked these 462 unmatched seller IDs against `order_items` and obtained the same result. This indicates that these sellers do not appear in the order-level seller data either. Therefore, the absence of seller information for these records is treated as a characteristic of the available data rather than as a data-cleaning error.

In [45]:
order_seller_ids = order_items["seller_id"].unique()

print(
    closed_deals["seller_id"].isin(order_seller_ids).sum()
)

380


In [46]:
print(
    closed_deals.loc[
        ~closed_deals["seller_id"].isin(order_seller_ids),
        "seller_id"
    ].nunique()
)

462


This merge adds seller-level attributes without changing the grain of `funnel_master`.

Missing seller attributes are expected for leads that do not have a corresponding seller record. For example, `seller_city` has 7,620 missing values. These missing values should not be interpreted as a general data-cleaning problem because many marketing leads did not result in a seller/closed deal, and some closed deals do not have a matching record in the seller/order data.

These missing values will be considered later during feature preparation depending on whether the relevant seller-level feature requires seller information.

In [48]:
funnel_master = funnel_master.merge(
    sellers,
    on="seller_id",
    how="left",
    validate="many_to_one"
)

print(funnel_master.shape)
print(funnel_master["mql_id"].nunique())
print(funnel_master["seller_id"].notna().sum())

(8000, 30)
8000
842


In [49]:
funnel_master["seller_city"].isna().sum()

np.int64(7620)

ŞURAYA BAK

In [52]:
df_master.isna().sum().sort_values(ascending=False)

review_comment_title             87889
review_comment_message           58665
order_delivered_customer_date     2965
estimated_delivery_gap_days       2965
delivery_time_days                2965
shipping_time_days                1797
order_delivered_carrier_date      1783
order_total_price                  775
order_total_freight                775
order_total_cost                   775
sellers_in_order                   775
is_free_shipping                   775
average_item_price                 775
items_in_order                     775
review_length                      768
review_score                       768
review_creation_date               768
review_answer_timestamp            768
response_time_hours                768
has_comment                        768
review_id                          768
approval_time_hours                160
order_approved_at                  160
payment_count                        1
total_payment_value                  1
order_id                 

In [53]:
print("Shape:", funnel_master.shape)
print("Unique leads:", funnel_master["mql_id"].nunique())

Shape: (8000, 30)
Unique leads: 8000


In [54]:
funnel_master.isna().sum().sort_values(ascending=False)

has_company                      7937
has_gtin                         7936
average_stock                    7934
declared_product_catalog_size    7931
seller_region                    7620
seller_state                     7620
seller_city                      7620
seller_zip_code_prefix           7620
seller_order_count               7620
lead_behaviour_profile           7335
business_type                    7168
lead_type                        7164
business_segment                 7159
sdr_id                           7158
won_date                         7158
seller_id                        7158
sr_id                            7158
declared_monthly_revenue         7158
won_year                         7158
won_month                        7158
won_quarter                      7158
has_declared_revenue             7158
origin                             60
contact_year                        0
landing_page_id                     0
contact_quarter                     0
contact_mont

The `products` table's grain is different from the `df_master` table's: "1 row = 1 product". Therefore, the `products` table cannot be directly integrated into the `df_master` table. A single order can contain multiple products, so directly joining product-level data duplicates order-level records and changes the level of detail in the `df_master` table.

In [55]:
print(products.shape)
print(products.columns.tolist())
print(products["product_id"].nunique())
print(products["product_id"].duplicated().sum())

(32951, 12)
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_volume_cm3', 'product_density_g_cm3', 'has_complete_listing']
32951
0


Before integration, I checked the relationship between `order_items` and `products`. All 32,951 different products in the `order_items` table have a corresponding record in the `products` table.

In [56]:
print(
    order_items["product_id"].isin(products["product_id"]).sum()
)

print(
    order_items["product_id"].nunique()
)

112650
32951


First, I merged the order_items and products tables. The purpose of this merge was to enrich each order item with relevant product attributes while preserving the original order item detail level. The merge did not duplicate order items.

In [57]:
order_items_products = order_items.merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one"
)

In [59]:
print(order_items_products.shape)
print(order_items_products["order_id"].nunique())

(112650, 27)
98666


These processes allow for the safe insertion of product-level information into order-level specifications without altering the level of detail in the df_master table, by converting product-level information into order-level specifications.

In [58]:
order_product_summary = (
    order_items_products
    .groupby("order_id")
    .agg(
        avg_product_weight_g=("product_weight_g", "mean"),
        avg_product_volume_cm3=("product_volume_cm3", "mean"),
        avg_product_density_g_cm3=("product_density_g_cm3", "mean"),
        avg_product_photos_qty=("product_photos_qty", "mean"),
        avg_product_name_length=("product_name_lenght", "mean"),
        avg_product_description_length=("product_description_lenght", "mean"),
        complete_listing_ratio=("has_complete_listing", "mean"),
        unique_product_categories=("product_category_name", "nunique")
    )
    .reset_index()
)

The resulting `order_product_summary` table shows:

1 row = 1 order.

The number of rows in the `df_master` table remained unchanged, confirming that the level of detail at the order level was preserved.

Why are the original product-level attributes still preserved?

The product-level attributes previously created in the `products` table are not replaced with these order-level summaries. Table and attribute selection should depend on the grain size being examined and the analytical question. They represent different analytical levels:

`products` → defines the attributes of an individual product
`df_master` → defines the overall product attributes of an order

In [60]:
df_master = df_master.merge(
    order_product_summary,
    on="order_id",
    how="left",
    validate="one_to_one"
)

BURADAN İTİBAREN MARKDOWN

In [62]:
print(df_master.shape)
print(df_master["order_id"].nunique())

(99441, 52)
99441


In [ ]:
df_master.isna().sum().sort_values(ascending=False)

review_comment_title              87889
review_comment_message            58665
order_delivered_customer_date      2965
estimated_delivery_gap_days        2965
delivery_time_days                 2965
avg_product_description_length     2164
avg_product_name_length            2164
avg_product_photos_qty             2164
shipping_time_days                 1797
order_delivered_carrier_date       1783
avg_product_weight_g                797
avg_product_density_g_cm3           797
avg_product_volume_cm3              791
order_total_cost                    775
sellers_in_order                    775
items_in_order                      775
is_free_shipping                    775
average_item_price                  775
order_total_price                   775
complete_listing_ratio              775
order_total_freight                 775
unique_product_categories           775
review_creation_date                768
review_answer_timestamp             768
response_time_hours                 768


In [65]:
print(df_master.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'purchase_year', 'purchase_month', 'purchase_day', 'purchase_hour', 'purchase_weekday', 'is_weekend', 'approval_time_hours', 'shipping_time_days', 'delivery_time_days', 'estimated_delivery_gap_days', 'is_late_delivery', 'purchase_period', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'is_repeat_customer', 'customer_region', 'items_in_order', 'order_total_price', 'order_total_freight', 'order_total_cost', 'sellers_in_order', 'is_free_shipping', 'average_item_price', 'total_payment_value', 'payment_count', 'review_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp', 'response_time_hours', 'has_comment', 'review_length', 'avg_product_weight_g', 'avg_product_volume_cm3', 'avg_product_density_g_cm

In [66]:
feature_audit = pd.DataFrame({
    "feature": df_master.columns,
    "dtype": df_master.dtypes.astype(str).values,
    "missing_count": df_master.isna().sum().values,
    "missing_pct": (df_master.isna().mean() * 100).values,
    "nunique": df_master.nunique(dropna=True).values
})

feature_audit

,feature,dtype,missing_count,missing_pct,nunique
0,order_id,str,0,0.000000,99441
1,customer_id,str,0,0.000000,99441
2,order_status,str,0,0.000000,8
3,order_purchase_timestamp,str,0,0.000000,98875
4,order_approved_at,str,160,0.160899,90733
5,order_delivered_carrier_date,str,1783,1.793023,81018
6,order_delivered_customer_date,str,2965,2.981668,95664
7,order_estimated_delivery_date,str,0,0.000000,459
8,purchase_year,int64,0,0.000000,3
9,purchase_month,int64,0,0.000000,12


In [67]:
customer_summary = (
    df_master
    .groupby("customer_unique_id")
    .agg(
        customer_order_count=("order_id", "nunique"),
        customer_total_spend=("order_total_price", "sum"),
        customer_avg_order_value=("order_total_price", "mean"),
        customer_total_freight=("order_total_freight", "sum"),
        customer_avg_freight=("order_total_freight", "mean"),
        customer_first_order_date=("order_purchase_timestamp", "min"),
        customer_last_order_date=("order_purchase_timestamp", "max")
    )
    .reset_index()
)

In [73]:
print(customer_summary.shape)
print(customer_summary["customer_unique_id"].nunique())

(96096, 9)
96096


In [74]:
customer_summary.head()

,customer_unique_id,customer_order_count,customer_total_spend,customer_avg_order_value,customer_total_freight,customer_avg_freight,customer_first_order_date,customer_last_order_date,customer_unique_categories
0,0000366f3b9a7992bf8c76cfdf3221e2,1,129.90,129.90,12.00,12.00,2018-05-10 10:56:27,2018-05-10 10:56:27,1.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,18.90,18.90,8.29,8.29,2018-05-07 11:11:27,2018-05-07 11:11:27,1.0
2,0000f46a3911fa3c0805444483337064,1,69.00,69.00,17.22,17.22,2017-03-10 21:05:03,2017-03-10 21:05:03,1.0
3,0000f6ccb0745a6a4b88665a16c9f078,1,25.99,25.99,17.63,17.63,2017-10-12 20:29:41,2017-10-12 20:29:41,1.0
4,0004aac84e0df4da2b147fca70cf8255,1,180.00,180.00,16.89,16.89,2017-11-14 19:45:42,2017-11-14 19:45:42,1.0


In [68]:
order_customer_map = df_master[
    ["order_id", "customer_unique_id"]
].drop_duplicates()

In [69]:
order_items_products_customer = order_items_products.merge(
    order_customer_map,
    on="order_id",
    how="left",
    validate="many_to_one"
)

In [71]:
customer_category_summary = (
    order_items_products_customer
    .groupby("customer_unique_id")
    .agg(
        customer_unique_categories=("product_category_name", "nunique")
    )
    .reset_index()
)

In [72]:
customer_summary = customer_summary.merge(
    customer_category_summary,
    on="customer_unique_id",
    how="left",
    validate="one_to_one"
)

In [75]:
df_master = df_master.merge(
    customer_summary,
    on="customer_unique_id",
    how="left",
    validate="many_to_one"
)

In [76]:
print("df_master shape:", df_master.shape)
print("Unique orders:", df_master["order_id"].nunique())
print("Unique customers:", df_master["customer_unique_id"].nunique())

df_master shape: (99441, 60)
Unique orders: 99441
Unique customers: 96096


In [77]:
print("Shape:", df_master.shape)
print("Unique orders:", df_master["order_id"].nunique())
print("Duplicate order_ids:", df_master["order_id"].duplicated().sum())

Shape: (99441, 60)
Unique orders: 99441
Duplicate order_ids: 0


In [78]:
customer_features = [
    "customer_order_count",
    "customer_total_spend",
    "customer_avg_order_value",
    "customer_total_freight",
    "customer_avg_freight",
    "customer_first_order_date",
    "customer_last_order_date",
    "customer_unique_categories"
]

df_master[customer_features].isna().sum()

customer_order_count            0
customer_total_spend            0
customer_avg_order_value      685
customer_total_freight          0
customer_avg_freight          685
customer_first_order_date       0
customer_last_order_date        0
customer_unique_categories    685
dtype: int64

In [79]:
df_master[
    [
        "customer_order_count",
        "customer_total_spend",
        "customer_avg_order_value",
        "customer_total_freight",
        "customer_avg_freight",
        "customer_unique_categories"
    ]
].describe()

,customer_order_count,customer_total_spend,customer_avg_order_value,customer_total_freight,customer_avg_freight,customer_unique_categories
count,99441.000000,99441.000000,98756.000000,99441.000000,98756.000000,98756.000000
mean,1.079223,146.228040,137.831302,24.422617,22.827877,1.048605
std,0.396154,223.701094,209.889449,24.682809,21.460615,0.247773
min,1.000000,0.000000,0.850000,0.000000,0.000000,1.000000
25%,1.000000,48.000000,46.990000,14.100000,13.920000,1.000000
50%,1.000000,89.900000,87.800000,17.670000,17.280000,1.000000
75%,1.000000,159.900000,149.900000,26.610000,24.260000,1.000000
max,17.000000,13440.000000,13440.000000,1794.960000,1794.960000,5.000000


In [81]:
df_master[
    df_master["customer_avg_order_value"].isna()
][
    [
        "customer_unique_id",
        "customer_order_count",
        "customer_total_spend",
        "customer_total_freight",
        "customer_avg_order_value",
        "customer_avg_freight",
        "customer_unique_categories"
    ]
].head(20)

,customer_unique_id,customer_order_count,customer_total_spend,customer_total_freight,customer_avg_order_value,customer_avg_freight,customer_unique_categories
266,41fc647b8c6bd979b1b6364b60471b50,1,0.0,0.0,NaN,NaN,NaN
586,0e634b16e4c585acbd7b2e8276ce6677,1,0.0,0.0,NaN,NaN,NaN
687,596ed6d7a35890b3fbac54ec01f69685,1,0.0,0.0,NaN,NaN,NaN
737,2349bbb558908e0955e98d47dacb7adb,1,0.0,0.0,NaN,NaN,NaN
1130,4fa4365000c7090fcb8cad5713c6d3db,1,0.0,0.0,NaN,NaN,NaN
1160,21c933c8dd97d088e64c50988c90ccf5,1,0.0,0.0,NaN,NaN,NaN
1579,bdc67efa33dd0c3228b91714ac6e363c,1,0.0,0.0,NaN,NaN,NaN
1826,45b1948a7554a397cc42c2ea55c54ab6,1,0.0,0.0,NaN,NaN,NaN
1868,c219f4ac1bc7f1aea33e6ab8885831e8,1,0.0,0.0,NaN,NaN,NaN
2029,a8dd81392e5eee5d979c629a76abec2a,1,0.0,0.0,NaN,NaN,NaN


In [82]:
df_master[
    df_master["customer_total_spend"].eq(0)
][
    [
        "order_id",
        "customer_unique_id",
        "order_status",
        "items_in_order",
        "order_total_price",
        "total_payment_value",
        "customer_total_spend",
        "customer_avg_order_value",
        "customer_unique_categories"
    ]
].head(20)

,order_id,customer_unique_id,order_status,items_in_order,order_total_price,total_payment_value,customer_total_spend,customer_avg_order_value,customer_unique_categories
266,8e24261a7e58791d10cb1bf9da94df5c,41fc647b8c6bd979b1b6364b60471b50,unavailable,NaN,NaN,84.00,0.0,NaN,NaN
586,c272bcd21c287498b4883c7512019702,0e634b16e4c585acbd7b2e8276ce6677,unavailable,NaN,NaN,97.68,0.0,NaN,NaN
687,37553832a3a89c9b2db59701c357ca67,596ed6d7a35890b3fbac54ec01f69685,unavailable,NaN,NaN,132.46,0.0,NaN,NaN
737,d57e15fb07fd180f06ab3926b39edcd2,2349bbb558908e0955e98d47dacb7adb,unavailable,NaN,NaN,134.38,0.0,NaN,NaN
1130,00b1cb0320190ca0daa2c88b35206009,4fa4365000c7090fcb8cad5713c6d3db,canceled,NaN,NaN,0.00,0.0,NaN,NaN
1160,2f634e2cebf8c0283e7ef0989f77d217,21c933c8dd97d088e64c50988c90ccf5,unavailable,NaN,NaN,615.53,0.0,NaN,NaN
1579,ee0db22a8e742b752914016708470ec8,bdc67efa33dd0c3228b91714ac6e363c,unavailable,NaN,NaN,167.82,0.0,NaN,NaN
1826,6ad57aecbae806a7e9cc2cdb6b380711,45b1948a7554a397cc42c2ea55c54ab6,unavailable,NaN,NaN,161.47,0.0,NaN,NaN
1868,df8282afe61008dc26c6c31011474d02,c219f4ac1bc7f1aea33e6ab8885831e8,canceled,NaN,NaN,139.96,0.0,NaN,NaN
2029,8d4c637f1accf7a88a4555f02741e606,a8dd81392e5eee5d979c629a76abec2a,canceled,NaN,NaN,66.44,0.0,NaN,NaN


In [83]:
df_master[
    df_master["customer_avg_order_value"].isna()
]["order_status"].value_counts()

order_status
unavailable    569
canceled       109
created          4
invoiced         2
shipped          1
Name: count, dtype: int64